# The Seven Layers of AI: Layer 5: Generative AI

This notebook contains every example program from **Chapter 6** of the study guide *The Seven Layers of AI*, in the order they appear. Each program comes with the explanation that introduces it in the guide, the output printed in the guide, what to notice in that output, and the Investigate tasks.

**How to use it:** run the cells from top to bottom. Each program is self-contained, so you can also run any single program on its own after the setup cell. Change the code freely: the Investigate tasks suggest where to start.

## Programs in this notebook

1. `l5_corpus.txt`: an n-gram language model
2. `l5_ngram.py`: an n-gram language model
3. `l5_bpe.py`: byte-pair encoding from scratch
4. `l5_embeddings.py`: embeddings from co-occurrence
5. `l5_mini_gpt.py`: a miniature GPT trained from scratch
6. `l5_sampling.py`: decoding strategies
7. `l5_rag.py`: the retrieval half of RAG
8. `l5_diffusion.py`: a diffusion model generating 2D data
9. `l5_llm_apis.py`: 6.11 Using Real Generative Models

## Setup

The programs in this notebook use: `numpy scikit-learn autograd`. If any are missing, uncomment the line below and run it once.

In [1]:
# %pip install numpy scikit-learn autograd
%matplotlib inline

## 1. An n-gram language model

**File:** `l5_corpus.txt`

Several programs in this chapter use the short original story below as training text. Save it as l5_corpus.txt in the same folder as the programs.

In [2]:
%%writefile l5_corpus.txt
the robot walked into the workshop and looked at the broken clock. the clock had stopped at midnight, and the robot wanted to know why. it opened the back of the clock and found a small gear with a missing tooth. the robot made a new gear from a piece of brass, fitted it carefully, and closed the case. the clock began to tick again. the next morning the engineer came into the workshop and heard the ticking. she smiled at the robot and said that it had done good work. the robot did not smile, because robots cannot smile, but it felt that the day had started well.
the engineer gave the robot a new task. a bicycle in the corner had a flat tyre and a loose chain. the robot lifted the bicycle onto the bench, removed the wheel, and found a sharp thorn in the tyre. it pulled out the thorn, patched the inner tube, and pumped the tyre until it was firm. then it tightened the chain and oiled the gears. the engineer rode the bicycle around the yard and said that it had never felt so smooth.
in the afternoon the robot tidied the workshop. it put the spanners in one drawer and the screwdrivers in another. it swept the floor and wiped the bench. when the sun went down the robot sat by the window and watched the lights of the town. it thought about the clock, the bicycle and the engineer, and it wondered what it would repair tomorrow.


Overwriting l5_corpus.txt


## 2. An n-gram language model

**File:** `l5_ngram.py`

In [3]:
"""
Layer 5 example: the simplest generative language model, an n-gram model.
 
File: l5_ngram.py
 
Purpose:
    A language model assigns probabilities to sequences of words. Using the
    chain rule of probability:
        P(w1, w2, ..., wn) = P(w1) * P(w2 | w1) * P(w3 | w1, w2) * ...
    An n-gram model ASSUMES each word depends only on the previous n-1 words.
    Large language models make the same next-token prediction, but condition
    on thousands of previous tokens using a transformer instead of a count table.
 
How it works:
    1. Read a short story (l5_corpus.txt), split it into words and hold back
       the last 15% as unseen test text.
    2. For n = 1, 2, 3 count which word follows each (n-1)-word context.
    3. Turn counts into smoothed probabilities and measure perplexity on the
       training text and on the unseen test text.
    4. Generate new text by repeatedly sampling a next word from the counts.
 
What to look for in the output:
    - Train perplexity falls as the context grows (56.5 -> 13.8 -> 12.5).
    - Test perplexity does not: the 3-gram scores 135.0, worse than the
      2-gram's 103.7 and even above uniform guessing (129), because most
      3-word contexts in the test text were never seen (data sparsity).
    - The 3-gram samples read smoothly because they mostly copy training text.
    - After 'the', the most likely next word is 'robot' (0.205).
 
Requires:
    Python 3 standard library only; l5_corpus.txt in the same folder.
"""
 
import math
import random
from collections import Counter, defaultdict
 
# Fix the random seed so the generated samples are the same on every run.
random.seed(3)
# Put spaces before full stops and commas so punctuation becomes its own "word"
# (a very crude tokeniser; real LLMs use subword tokens, see l5_bpe.py).
words = open("l5_corpus.txt", encoding="utf-8").read().replace(
    ".", " .").replace(",", " ,").split()
# Hold back the final 15% of the text to test how well the model generalises.
split = int(len(words) * 0.85)
train, test = words[:split], words[split:]
vocab = set(words)
print(f"{len(words)} words, vocabulary {len(vocab)}, "
      f"train {len(train)}, test {len(test)}\n")
 
 
# ---- Counting n-grams -----------------------------------------------------
# counts[context][next_word] = how many times next_word followed that context.
# For n=1 the context is the empty tuple, so we just count word frequencies.
# Dividing these counts by the context total estimates P(word | context).
def build(n):
    """Count how often each word follows each context of n-1 words."""
    counts = defaultdict(Counter)
    for i in range(len(train) - n + 1):
        context = tuple(train[i:i+n-1])           # the n-1 words before
        counts[context][train[i+n-1]] += 1        # the word that followed
    return counts
 
 
# ---- Smoothing --------------------------------------------------------------
# Without smoothing, any word never seen after a context gets probability 0,
# and one zero makes the probability of a whole test sentence 0 (log = -inf).
def prob(counts, context, word, k=0.1):
    """Add-k smoothing: pretend every word was seen k extra times in every context,
    so unseen combinations get a small probability instead of zero."""
    c = counts.get(tuple(context), Counter())
    # numerator: real count + k; denominator: context total + k for every word
    return (c[word] + k) / (sum(c.values()) + k * len(vocab))
 
 
# ---- Perplexity -------------------------------------------------------------
# By the chain rule, log P(sequence) is the sum of log P(word | context).
# Averaging the negative log gives the cross-entropy loss that LLMs also
# minimise; exp() of it turns it back into an "effective number of choices".
def perplexity(counts, n, seq):
    """Perplexity = exp(average negative log probability).
    Interpretation: the model is as uncertain as if choosing uniformly among
    this many words at each step. Lower is better."""
    total = 0.0
    for i in range(n - 1, len(seq)):
        total -= math.log(prob(counts, seq[i-n+1:i], seq[i]))
    return math.exp(total / (len(seq) - n + 1))
 
 
# ---- Generation ---------------------------------------------------------------
# Generating text is just repeated next-word prediction: look up the last n-1
# words, sample a follower in proportion to its count, append, repeat.
# This autoregressive loop is exactly how an LLM writes its answers.
def generate(counts, n, start, length=20, temperature=1.0):
    out = list(start)
    for _ in range(length):
        # context = last n-1 words generated so far (empty for a unigram)
        c = counts.get(tuple(out[-(n-1):]) if n > 1 else (), None)
        if not c:
            break                                 # dead end: context never seen
        cands = list(c)
        # temperature: raise counts to 1/T. T<1 sharpens, T>1 flattens the distribution
        weights = [c[w] ** (1 / temperature) for w in cands]
        out.append(random.choices(cands, weights)[0])
    return " ".join(out)
 
 
# ---- Compare unigram, bigram and trigram models -------------------------------
# Longer contexts give sharper predictions on training text but meet far more
# unseen contexts on new text: the classic bias/variance trade-off.
for n in (1, 2, 3):
    counts = build(n)
    # start the sample with enough words to fill the first context
    start = ["the", "robot"][:max(0, n-1)] if n > 1 else []
    print(f"{n}-gram  train perplexity {perplexity(counts, n, train):7.1f}   "
          f"test perplexity {perplexity(counts, n, test):7.1f}")
    print(f"        sample: {generate(counts, n, start or ['the'])}\n")
# A model that knows nothing picks uniformly, so its perplexity = vocabulary size.
print(f"Uniform guessing would give perplexity {len(vocab)}.")
print(
    "Longer contexts fit the training text far better "
    "but generalise worse to new text:")
print(
    "most 3-word contexts in the test set were never seen during training "
    "(data sparsity).")
 
# ---- Inspect one conditional distribution ------------------------------------
# P(next word | 'the') estimated directly from bigram counts (no smoothing).
bigram = build(2)
print("\nNext-word distribution after 'the':")
total = sum(bigram[('the',)].values())
for w, c in bigram[('the',)].most_common(6):
    print(f"  {w:10s} {c/total:.3f}")

286 words, vocabulary 129, train 243, test 43

1-gram  train perplexity    56.5   test perplexity   133.9
        sample: the and a . from case the the chain and and floor it chain it again the tick sharp found cannot

2-gram  train perplexity    13.8   test perplexity   103.7
        sample: the bench , but it had stopped at the tyre until it swept the inner tube , removed the robot wanted

3-gram  train perplexity    12.5   test perplexity   135.0
        sample: the robot tidied the workshop and heard the ticking . she smiled at the robot tidied the workshop . it swept the

Uniform guessing would give perplexity 129.
Longer contexts fit the training text far better but generalise worse to new text:
most 3-word contexts in the test set were never seen during training (data sparsity).

Next-word distribution after 'the':
  robot      0.205
  workshop   0.077
  clock      0.077
  engineer   0.077
  bicycle    0.051
  tyre       0.051


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
286 words, vocabulary 129, train 243, test 43
 
1-gram  train perplexity    56.5   test perplexity   133.9
        sample: the and a . from case the the chain and and floor it chain it again the tick sharp found cannot
 
2-gram  train perplexity    13.8   test perplexity   103.7
        sample: the bench , but it had stopped at the tyre until it swept the inner tube , removed the robot wanted
 
3-gram  train perplexity    12.5   test perplexity   135.0
        sample: the robot tidied the workshop and heard the ticking . she smiled at the robot tidied the workshop . it swept the
 
Uniform guessing would give perplexity 129.
Longer contexts fit the training text far better but generalise worse to new text:
most 3-word contexts in the test set were never seen during training (data sparsity).
 
Next-word distribution after 'the':
  robot      0.205
  workshop   0.077
  clock      0.077
  engineer   0.077
  bicycle    0.051
  tyre       0.051
```

**What the output shows**

The bigram model assigns much higher probability to its training text than the unigram model, and its samples look locally fluent. But on unseen test text all three models are barely better than uniform guessing, and the trigram does worse than the bigram, because almost every three-word context in the test text is new. Overcoming this sparsity is precisely what neural language models do.

**Investigate**

1. Train the models on a full public domain novel from Project Gutenberg. How do perplexities change with far more data?
2. Try smoothing values k = 1, 0.1 and 0.001. Which gives the lowest test perplexity?
3. Generate with temperature 0.5 and 2.0 using the bigram model and describe the difference.

## 3. Byte-pair encoding from scratch

**File:** `l5_bpe.py`

In [4]:
"""
Layer 5 example: byte-pair encoding (BPE) tokenisation from scratch.
 
File: l5_bpe.py
 
Purpose:
    LLMs do not read whole words or single letters, but TOKENS: frequently
    occurring pieces of text. BPE (Sennrich et al., 2016, adapted from a 1994
    compression algorithm) builds a vocabulary by repeatedly MERGING the most
    frequent adjacent pair of symbols into a new symbol.
    Common words become single tokens; rare words split into familiar pieces,
    so the model never meets a completely unknown word.
 
How it works:
    1. Split every training word into single characters plus an end marker '_'.
    2. Count every adjacent pair of symbols, weighted by word frequency.
    3. Merge the most frequent pair into one new symbol; repeat 20 times.
    4. Tokenise new words by replaying the learned merges in the same order.
 
What to look for in the output:
    - The first merge is 'l' + 'o' -> 'lo', and by merge 8 'lower_' is a
      single symbol: frequent pieces are learned first.
    - 'lowest' becomes 1 token, 'slower' becomes ['s', 'lower_'].
    - Unseen words still tokenise: 'xylophone' -> 8 tokens and 'bicycles'
      falls back to 9 single-character tokens.
 
Requires:
    Python 3 standard library only.
"""
 
from collections import Counter
 
# A tiny training corpus (repeated 3 times so frequencies are clearer).
corpus = ("low lower lowest newer newest wider widest slow slower slowest "
          "the robot repaired the lower clock and the newest bicycle " * 3).split()
 
 
# ---- Count adjacent symbol pairs ----------------------------------------------
# word_freqs maps a word, stored as a tuple of symbols, to how often it occurs.
# A pair inside a word that appears 6 times counts 6 times.
def get_pairs(word_freqs):
    pairs = Counter()
    for word, freq in word_freqs.items():
        for a, b in zip(word, word[1:]):          # every neighbouring pair
            pairs[(a, b)] += freq
    return pairs
 
 
# ---- Apply one merge -----------------------------------------------------------
# Scan each word left to right and glue every occurrence of `pair` together,
# e.g. ('l','o','w') with pair ('l','o') -> ('lo','w').
def merge(word_freqs, pair):
    merged = {}
    for word, freq in word_freqs.items():
        new, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                # replace the pair with one symbol and skip past both halves
                new.append(word[i] + word[i+1]); i += 2
            else:
                new.append(word[i]); i += 1
        merged[tuple(new)] = freq
    return merged
 
 
# ---- Training: learn 20 merges ---------------------------------------------------
# Start: every word is a sequence of characters, '_' marks the end of a word
# The '_' lets the tokeniser tell a word ending ('st_') from a word middle ('st').
word_freqs = Counter(tuple(w) + ("_",) for w in corpus)
merges = []                     # the ordered merge list IS the learned tokeniser
for step in range(20):
    pairs = get_pairs(word_freqs)
    best = pairs.most_common(1)[0][0]             # most frequent adjacent pair
    merges.append(best)
    word_freqs = merge(word_freqs, best)
    # print only the first 8 merges and the last one to keep the output short
    if step < 8 or step == 19:
        print(f"merge {step+1:2d}: {best[0]!r} + {best[1]!r} -> {best[0]+best[1]!r}")
 
 
# ---- Tokenising new text ------------------------------------------------------------
# Every word starts as characters, which are always in the vocabulary, so no
# word is ever "unknown". Merges then build it back up into the largest pieces
# the tokeniser knows. Real LLM tokenisers start from bytes and learn ~50,000
# to 200,000 merges, so common words become one token and rare ones a few.
def tokenize(word):
    """Apply the learned merges, in the order they were learned, to a new word."""
    symbols = list(word) + ["_"]
    for pair in merges:
        # reuse merge() on a one-word dictionary and take back the merged word
        symbols = list(merge({tuple(symbols): 1}, pair).keys())[0]
    return list(symbols)
 
 
# Some words below were in the training corpus; 'newly', 'bicycles' and
# 'xylophone' were not. Fewer tokens per word = more text fits in the context.
print("\nTokenising words, including ones never seen in training:")
for w in ["lowest", "slower", "widest", "robot", "newly", "bicycles", "xylophone"]:
    toks = tokenize(w)
    print(f"  {w:10s} -> {toks}  ({len(toks)} tokens)")
print("\nUnseen words still tokenise, "
      "falling back to smaller pieces or single characters.")

merge  1: 'l' + 'o' -> 'lo'
merge  2: 'w' + 'e' -> 'we'
merge  3: 't' + '_' -> 't_'
merge  4: 'lo' + 'we' -> 'lowe'
merge  5: 'r' + '_' -> 'r_'
merge  6: 's' + 't_' -> 'st_'
merge  7: 'e' + '_' -> 'e_'
merge  8: 'lowe' + 'r_' -> 'lower_'
merge 20: 'r' + 'e' -> 're'

Tokenising words, including ones never seen in training:
  lowest     -> ['lowest_']  (1 tokens)
  slower     -> ['s', 'lower_']  (2 tokens)
  widest     -> ['wide', 'st_']  (2 tokens)
  robot      -> ['r', 'o', 'b', 'o', 't_']  (5 tokens)
  newly      -> ['ne', 'w', 'l', 'y', '_']  (5 tokens)
  bicycles   -> ['b', 'i', 'c', 'y', 'c', 'l', 'e', 's', '_']  (9 tokens)
  xylophone  -> ['x', 'y', 'lo', 'p', 'h', 'o', 'n', 'e_']  (8 tokens)

Unseen words still tokenise, falling back to smaller pieces or single characters.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
merge  1: 'l' + 'o' -> 'lo'
merge  2: 'w' + 'e' -> 'we'
merge  3: 't' + '_' -> 't_'
merge  4: 'lo' + 'we' -> 'lowe'
merge  5: 'r' + '_' -> 'r_'
merge  6: 's' + 't_' -> 'st_'
merge  7: 'e' + '_' -> 'e_'
merge  8: 'lowe' + 'r_' -> 'lower_'
merge 20: 'r' + 'e' -> 're'
 
Tokenising words, including ones never seen in training:
  lowest     -> ['lowest_']  (1 tokens)
  slower     -> ['s', 'lower_']  (2 tokens)
  widest     -> ['wide', 'st_']  (2 tokens)
  robot      -> ['r', 'o', 'b', 'o', 't_']  (5 tokens)
  newly      -> ['ne', 'w', 'l', 'y', '_']  (5 tokens)
  bicycles   -> ['b', 'i', 'c', 'y', 'c', 'l', 'e', 's', '_']  (9 tokens)
  xylophone  -> ['x', 'y', 'lo', 'p', 'h', 'o', 'n', 'e_']  (8 tokens)
 
Unseen words still tokenise, falling back to smaller pieces or single characters.
```

**Investigate**

1. Run 100 merges on the chapter text of this guide and inspect which tokens are created first.
2. Why does "robot" split into five tokens here when real tokenisers treat it as one? What would change with a larger corpus?
3. Estimate how many tokens a 1,000-word essay would contain using the rule of thumb that 100 tokens is about 75 English words.

## 4. Embeddings from co-occurrence

**File:** `l5_embeddings.py`

In [5]:
"""
Layer 5 example: word embeddings learned from co-occurrence.
 
File: l5_embeddings.py
 
Purpose:
    'You shall know a word by the company it keeps' (Firth, 1957).
    Words used in similar contexts should get similar vectors. Every LLM
    begins by turning each token into such a vector (an embedding), and RAG
    systems search documents by comparing embedding vectors.
 
How it works (a simple relative of word2vec and GloVe):
  1. count how often each word appears near each other word
  2. convert counts to positive pointwise mutual information (PPMI)
  3. compress the matrix with singular value decomposition (SVD)
Each word becomes a short dense vector; similarity = cosine of the angle.
  4. list each test word's nearest neighbours and solve analogies such as
     man : king = woman : ? by vector arithmetic.
 
What to look for in the output:
    - 43 words are squeezed into 10-dimensional vectors.
    - 'king' is closest to 'queen' (1.0), 'princess' and 'prince' (0.94).
    - man : king = woman : queen and man : boy = woman : girl come out right.
    - france : paris = england : pet FAILS, because in this tiny corpus
      'paris' shares its 'X is the ...' context with 'pet' (similarity 0.98).
 
Requires:
    NumPy.
"""
 
import numpy as np
from collections import Counter
 
# A tiny hand-written corpus: royalty, animals and capital cities.
# .split() turns it into one long list of word tokens.
sentences = """
the king rules the kingdom . the queen rules the kingdom .
the king is a man . the queen is a woman . the prince is a young man .
the princess is a young woman . the man walks to the town . \
the woman walks to the town .
the king wears a crown . the queen wears a crown . the prince will become king .
the princess will become queen . a boy is a young man . a girl is a young woman .
the cat chases the mouse . the dog chases the cat . the cat drinks milk .
the dog eats meat . the mouse eats cheese . the cat is a pet . the dog is a pet .
paris is the capital of france . london is the capital of england .
berlin is the capital of germany . rome is the capital of italy .
france is a country . england is a country . germany is a country . italy is a country .
""".split()
 
# Give every distinct word a row/column number in the matrices below.
vocab = sorted(set(sentences))
idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
 
# ---- 1. Co-occurrence counts within a window of 2 words either side ----------------
# C[w, c] = how often context word c appears within 2 positions of word w.
# Shape (V, V). Words with similar rows are used in similar company.
C = np.zeros((V, V))
for i, w in enumerate(sentences):
    for j in range(max(0, i - 2), min(len(sentences), i + 3)):
        if i != j:
            C[idx[w], idx[sentences[j]]] += 1
 
# ---- 2. PPMI: how much MORE often do two words co-occur than chance predicts? --------
# Raw counts are dominated by frequent words such as "the" and ".", which sit
# next to everything. PMI = log( P(w, c) / (P(w) * P(c)) ) corrects for this:
# it is positive when two words meet more often than if they were independent.
total = C.sum()
# P(w): row totals, shape (V, 1); P(c): column totals, shape (1, V)
p_w = C.sum(1, keepdims=True) / total
p_c = C.sum(0, keepdims=True) / total
# pairs never seen together give log(0) = -inf; silence the warning
with np.errstate(divide="ignore"):
    pmi = np.log((C / total) / (p_w * p_c))
ppmi = np.maximum(pmi, 0)                      # keep only positive associations
 
# ---- 3. SVD compresses 43 dimensions (one per word) to 10 dense dimensions ----
# SVD factorises ppmi = U * diag(S) * Vt, with S sorted largest first.
# Keeping only the top 10 directions keeps the strongest patterns and drops
# noise, giving each word a dense 10-number vector (rows of E, shape (V, 10)).
U, S, _ = np.linalg.svd(ppmi)
E = U[:, :10] * S[:10]
# scale every vector to unit length, so a dot product equals the cosine
E = E / np.linalg.norm(E, axis=1, keepdims=True)
print(f"vocabulary {V} words, embedding size {E.shape[1]}\n")
 
 
# ---- Nearest neighbours by cosine similarity ------------------------------------
# Cosine = 1 means same direction (used identically), 0 means unrelated.
# Very common filler words are skipped so the neighbours are more informative.
def nearest(word, k=4):
    sims = E @ E[idx[word]]                       # cosine with every word, shape (V,)
    order = [i for i in np.argsort(-sims)
             if vocab[i] != word and vocab[i] not in {"the", "a", "is", "."}]
    return [(vocab[i], round(float(sims[i]), 2)) for i in order[:k]]
 
 
for w in ["king", "cat", "france", "woman", "paris"]:
    print(f"most similar to {w:7s}: {nearest(w)}")
 
 
# ---- Analogies by vector arithmetic ----------------------------------------------
# If b - a captures a relationship (e.g. "royal version of"), adding it to c
# should land near the word that stands in that relationship to c.
def analogy(a, b, c):
    """a is to b as c is to ?   computed as  vector(b) - vector(a) + vector(c)"""
    target = E[idx[b]] - E[idx[a]] + E[idx[c]]
    sims = E @ (target / np.linalg.norm(target))
    # take the most similar word that is not one of the three inputs
    for i in np.argsort(-sims):
        if vocab[i] not in {a, b, c}:
            return vocab[i]
 
 
print("\nAnalogies by vector arithmetic:")
for a, b, c in [("man", "king", "woman"), ("king", "queen", "prince"),
                ("man", "boy", "woman"), ("france", "paris", "england")]:
    print(f"  {a} : {b} = {c} : {analogy(a, b, c)}")
print("\nNo meanings were programmed: "
      "the structure comes purely from which words appear together.")
print("The capital-city results fail: in this tiny corpus 'paris' appears in the same")
print("'X is the ...' pattern as 'pet', 'boy' and 'girl'. "
      "Real embeddings use billions of words.")

vocabulary 43 words, embedding size 10

most similar to king   : [('queen', 1.0), ('princess', 0.94), ('prince', 0.94), ('kingdom', 0.9)]
most similar to cat    : [('drinks', 0.79), ('milk', 0.79), ('mouse', 0.48), ('dog', 0.47)]
most similar to france : [('italy', 0.98), ('england', 0.82), ('germany', 0.82), ('of', 0.82)]
most similar to woman  : [('man', 1.0), ('walks', 0.97), ('town', 0.86), ('young', 0.62)]
most similar to paris  : [('pet', 0.98), ('girl', 0.95), ('boy', 0.95), ('crown', 0.74)]

Analogies by vector arithmetic:
  man : king = woman : queen
  king : queen = prince : princess
  man : boy = woman : girl
  france : paris = england : pet

No meanings were programmed: the structure comes purely from which words appear together.
The capital-city results fail: in this tiny corpus 'paris' appears in the same
'X is the ...' pattern as 'pet', 'boy' and 'girl'. Real embeddings use billions of words.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
vocabulary 43 words, embedding size 10
 
most similar to king   : [('queen', 1.0), ('princess', 0.94), ('prince', 0.94), ('kingdom', 0.9)]
most similar to cat    : [('drinks', 0.79), ('milk', 0.79), ('mouse', 0.48), ('dog', 0.47)]
most similar to france : [('italy', 0.98), ('england', 0.82), ('germany', 0.82), ('of', 0.82)]
most similar to woman  : [('man', 1.0), ('walks', 0.97), ('town', 0.86), ('young', 0.62)]
most similar to paris  : [('pet', 0.98), ('girl', 0.95), ('boy', 0.95), ('crown', 0.74)]
 
Analogies by vector arithmetic:
  man : king = woman : queen
  king : queen = prince : princess
  man : boy = woman : girl
  france : paris = england : pet
 
No meanings were programmed: the structure comes purely from which words appear together.
The capital-city results fail: in this tiny corpus 'paris' appears in the same
'X is the ...' pattern as 'pet', 'boy' and 'girl'. Real embeddings use billions of words.
```

**What the output shows**

Royalty, people, animals and countries form separate clusters, and three of the four analogies succeed, even though no meanings were programmed. The capital city failure is instructive: with a corpus of a few hundred words, "paris" shares its grammatical pattern with unrelated words. Structure emerges from data, and weak data gives weak structure.

**Investigate**

1. Add ten sentences about capital cities in varied grammatical forms. Does the france : paris = england : ? analogy now succeed?
2. Change the window size from 2 to 1 and to 4. How do the nearest neighbours change?
3. Add sentences such as "the doctor is a man" and "the nurse is a woman". Test whether the embeddings now associate occupations with gender, and discuss the implications.

## 5. A miniature GPT trained from scratch

**File:** `l5_mini_gpt.py`

The following program implements a complete decoder-only transformer, with token and positional embeddings, causal multi-head self-attention, feed-forward layers, residual connections, layer normalisation and the Adam optimiser. It uses the small autograd library, which differentiates ordinary NumPy code automatically (Maclaurin et al., 2015), and trains in under a minute on a CPU. Install it with pip install autograd. It uses the l5_corpus.txt file from Section 6.2; any plain text file can be substituted.

In [6]:
"""
Layer 5 example: a miniature GPT-style language model, trained from scratch.
 
File: l5_mini_gpt.py
 
Purpose:
    This is the SAME architecture family as ChatGPT, Claude and Gemini, shrunk
    by a factor of millions so it trains on a laptop CPU in a few minutes.
    Watching its samples improve shows how next-token prediction alone turns
    random letters into words and sentences.
 
How it works:
  character tokens -> token + position embeddings
  -> 2 transformer blocks (causal multi-head self-attention + MLP,
     residuals, layer norm)
  -> linear layer -> softmax over the next character
Objective: predict the next token (self-supervised; the text labels itself).
Training: 1,500 steps of the Adam optimiser on random 32-character windows,
printing the loss and a generated sample at steps 1, 250, 750 and 1500.
 
We use the small 'autograd' library (pip install autograd) so that NumPy code
is differentiated automatically, exactly as PyTorch would do.
 
What to look for in the output:
    - 103,936 parameters (a frontier LLM has hundreds of billions).
    - The loss starts at 3.222, close to the 3.258 of uniform guessing over
      26 characters, and falls to 0.855, 0.234 and finally 0.172.
    - The step-1 sample is random letters; by step 250 there are word-like
      shapes; by step 1500 it writes whole phrases from the story, such as
      'it had done good work'. With such a small corpus it mostly memorises.
    - The times in brackets, e.g. (48s), will differ on other machines.
 
Requires:
    NumPy and autograd; l5_corpus.txt in the same folder.
"""
 
import time
import autograd.numpy as np          # a drop-in NumPy that records operations
from autograd import grad
import numpy as onp                  # ordinary NumPy for data handling
 
rng = onp.random.default_rng(0)                  # fixed seed: repeatable run
# ---- Data: character-level tokens ---------------------------------------------
# Real LLMs use BPE subword tokens (see l5_bpe.py); single characters keep
# the vocabulary tiny. Each character is mapped to an integer id.
text = open("l5_corpus.txt", encoding="utf-8").read().replace("\n", " ")  # one line
chars = sorted(set(text))
V = len(chars)
# stoi = "string to integer": character -> token id
stoi = {c: i for i, c in enumerate(chars)}
# the whole corpus as a 1D array of token ids, shape (1342,)
data = onp.array([stoi[c] for c in text])
print(f"corpus {len(text)} characters, vocabulary {V} tokens")
 
# ---- Hyperparameters (a real LLM: context 100k+, d_model 10k+, ~100 layers) ----
# T = how many previous tokens the model can see; D = length of every token's
# vector; HEADS = parallel attention patterns per block; LR = learning rate.
T, D, HEADS, LAYERS = 32, 64, 4, 2       # context length, model width, heads, blocks
BATCH, STEPS, LR = 24, 1500, 3e-3
 
 
# ---- Parameters ------------------------------------------------------------------
# Small random starting weights. Learning = adjusting these numbers.
def init(shape, scale=0.02):
    return rng.normal(0, scale, shape)
 
 
# tok: one learned vector per token, shape (V, D)
# pos: one learned vector per POSITION, shape (T, D). Attention on its own
#      ignores word order, so position vectors tell the model where each
#      token sits in the window.
# out: maps a final D-vector to V scores, one per possible next character
params = {"tok": init((V, D)), "pos": init((T, D)), "out": init((D, V))}
for l in range(LAYERS):
    params.update({
        f"ln1_{l}": onp.ones(D), f"ln2_{l}": onp.ones(D),
        # query, key and value projections in one matrix
        f"qkv_{l}": init((D, 3 * D)),
        f"proj_{l}": init((D, D)),            # mixes the attention heads back together
        # feed-forward MLP: widen to 4*D, apply ReLU, shrink back to D
        f"fc1_{l}": init((D, 4 * D)), f"fc2_{l}": init((4 * D, D)),
    })
print("parameters:", sum(p.size for p in params.values()))
 
 
# ---- Layer normalisation -------------------------------------------------------
# Rescale each token's vector to mean 0 and variance 1 (then multiply by a
# learned gain). Keeping values in a steady range makes deep networks train.
def layer_norm(x, gain):
    mu = np.mean(x, axis=-1, keepdims=True)
    var = np.mean((x - mu) ** 2, axis=-1, keepdims=True)
    return gain * (x - mu) / np.sqrt(var + 1e-5)
 
 
# causal mask: no peeking at the future. Lower-triangular (T, T) matrix of 1s:
# row i has 1s in columns 0..i, so position i may only attend to itself and
# earlier positions. Without it, predicting the next token would be cheating.
MASK = onp.tril(onp.ones((T, T)))
 
 
# ---- The transformer forward pass ----------------------------------------------
# idx: token ids, shape (B, t) = (batch size, sequence length).
# Returns next-token scores (logits) for EVERY position at once, shape (B, t, V):
# position i predicts token i+1, so one window gives t training examples.
def forward(p, idx):
    B, t = idx.shape
    # (B, t, D) token + position embeddings
    x = p["tok"][idx] + p["pos"][:t]
    hd = D // HEADS                               # width of each head (16)
    for l in range(LAYERS):
        # ---- causal multi-head self-attention (pre-norm, with residual) ----
        h = layer_norm(x, p[f"ln1_{l}"])
        # query = "what am I looking for", key = "what do I contain",
        # value = "what I pass on if chosen"; each (B, t, D)
        q, k, v = np.split(h @ p[f"qkv_{l}"], 3, axis=-1)
        # cut D into HEADS pieces so each head can attend to different things
        split = lambda z: np.transpose(np.reshape(z, (B, t, HEADS, hd)), (0, 2, 1, 3))
        q, k, v = split(q), split(k), split(v)    # (B, heads, t, hd)
        # similarity of every query with every key: (B, heads, t, t), scaled by
        # sqrt(hd) so the softmax does not become too sharp
        scores = q @ np.transpose(k, (0, 1, 3, 2)) / np.sqrt(hd)
        # future positions get -1e9, so after softmax their weight is 0
        scores = np.where(MASK[:t, :t] == 1, scores, -1e9)
        # softmax over each row -> attention weights that sum to 1
        scores = scores - np.max(scores, axis=-1, keepdims=True)
        w = np.exp(scores); w = w / np.sum(w, axis=-1, keepdims=True)
        # weighted average of the values, heads glued back: (B, t, D)
        att = np.reshape(np.transpose(w @ v, (0, 2, 1, 3)), (B, t, D))
        x = x + att @ p[f"proj_{l}"]              # residual connection
        # ---- feed-forward network (with residual) ----
        h = layer_norm(x, p[f"ln2_{l}"])
        # attention MIXES information between positions; this MLP then
        # processes each position on its own (ReLU = max(z, 0))
        x = x + np.maximum(h @ p[f"fc1_{l}"], 0) @ p[f"fc2_{l}"]
    return x @ p["out"]                           # (B, t, V) scores for the next token
 
 
# ---- Next-token loss --------------------------------------------------------------
# Cross-entropy: the average of -log P(correct next character). It is 0 for a
# perfect model and log(V) = 3.258 for uniform guessing. This is the same
# loss (and the same perplexity idea) as in l5_ngram.py.
def loss_fn(p, xb, yb):
    logits = forward(p, xb)
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    # log-softmax: log probabilities for every character, (B, t, V)
    logp = logits - np.log(np.sum(np.exp(logits), axis=-1, keepdims=True))
    B, t = yb.shape
    # pick out the log probability of the TRUE next character, (B, t)
    picked = logp[onp.arange(B)[:, None], onp.arange(t)[None, :], yb]
    # cross-entropy on the true next characters
    return -np.mean(picked)
 
 
# grad() builds a function returning d(loss)/d(parameter) for every
# parameter (backpropagation done automatically).
grad_fn = grad(loss_fn)
# Adam keeps two running averages per parameter: m (average gradient, a kind
# of momentum) and s (average squared gradient, used to scale each step).
m = {k: onp.zeros_like(v) for k, v in params.items()}
s = {k: onp.zeros_like(v) for k, v in params.items()}
 
 
# ---- Training batches ---------------------------------------------------------------
# Pick BATCH random windows of T characters. The target for each position is
# simply the next character in the text: no human labelling is needed.
def batch():
    starts = rng.integers(0, len(data) - T - 1, BATCH)
    xb = onp.stack([data[i:i+T] for i in starts])
    # targets = inputs shifted by one
    yb = onp.stack([data[i+1:i+T+1] for i in starts])
    return xb, yb
 
 
# ---- Generation ---------------------------------------------------------------------
# Autoregressive sampling: predict, sample, append, repeat. Only the last T
# tokens fit in the context window; older ones are forgotten.
def generate(p, prompt, n, temperature=0.8):
    idx = [stoi[c] for c in prompt]
    for _ in range(n):
        ctx = onp.array([idx[-T:]])
        # scores for the token after the last position, sharpened by T < 1
        logits = forward(p, ctx)[0, -1] / temperature
        probs = onp.exp(logits - logits.max()); probs /= probs.sum()
        # SAMPLE the next token, append, repeat
        idx.append(int(rng.choice(V, p=probs)))
    return "".join(chars[i] for i in idx)
 
 
# ---- Training loop ----------------------------------------------------------------
# Each step: sample a batch, compute gradients, nudge every parameter downhill.
start = time.time()
for step in range(1, STEPS + 1):
    xb, yb = batch()
    g = grad_fn(params, xb, yb)
    # Adam optimiser. The (1 - 0.9 ** step) terms correct the bias of m and s
    # starting at zero. Each parameter moves by about LR in the direction that
    # lowers the loss, whatever the raw size of its gradient.
    for k in params:
        m[k] = 0.9 * m[k] + 0.1 * g[k]            # momentum: smoothed gradient
        s[k] = 0.999 * s[k] + 0.001 * g[k] ** 2   # smoothed squared gradient
        params[k] -= LR * (m[k] / (1 - 0.9 ** step)) / (
            onp.sqrt(s[k] / (1 - 0.999 ** step)) + 1e-8)
    # report progress at a few checkpoints (the loss is on the current batch)
    if step in (1, 250, 750, 1500):
        print(f"\nstep {step:4d}  loss {loss_fn(params, xb, yb):.3f}  "
              f"({time.time() - start:.0f}s)")
        print("  sample:", repr(generate(params, "the robot ", 80)))
 
# The baseline: a model that has learned nothing gives every character
# probability 1/V, so its loss is log(V).
print(f"\nA uniform guess over {V} characters would have loss {onp.log(V):.3f}.")

corpus 1342 characters, vocabulary 26 tokens
parameters: 103936



step    1  loss 3.222  (0s)


  sample: 'the robot kiaykyipahrsmctvombi .glprawmaxwl l eiigtmsbxivyihdkaus,bdnegls wx,hwe dli k.wiw'



step  250  loss 0.855  (16s)
  sample: 'the robot as hop han sted the clock. an whed beaft he clockshtng. ile, beclosed the clock '



step  750  loss 0.234  (48s)
  sample: 'the robot made a nther. it thorn in the afternoon the robose chock. the clock and found th'



step 1500  loss 0.172  (98s)
  sample: 'the robot and said that it had done good work. the robot did not smile, because robots can'

A uniform guess over 26 characters would have loss 3.258.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
corpus 1342 characters, vocabulary 26 tokens
parameters: 103936
 
step    1  loss 3.222  (0s)
  sample: 'the robot kiaykyipahrsmctvombi .glprawmaxwl l eiigtmsbxivyihdkaus,bdnegls wx,hwe dli k.wiw'
 
step  250  loss 0.855  (8s)
  sample: 'the robot as hop han sted the clock. an whed beaft he clockshtng. ile, beclosed the clock '
 
step  750  loss 0.234  (24s)
  sample: 'the robot made a nther. it thorn in the afternoon the robose chock. the clock and found th'
 
step 1500  loss 0.172  (48s)
  sample: 'the robot and said that it had done good work. the robot did not smile, because robots can'
 
A uniform guess over 26 characters would have loss 3.258.
```

**What the output shows**

The progression mirrors what happens in real LLMs, compressed into a minute: random characters, then word-like fragments, then plausible phrases. There is also an important warning in the final sample. With about 100,000 parameters and only 1,342 characters of training text, the model has largely memorised its corpus and reproduces sentences verbatim. Real LLMs are trained on far more text than they can memorise, but they can still reproduce passages that appear many times in their training data, which has consequences for privacy and copyright (Carlini et al., 2021).

**Investigate**

1. Hold back the last 20% of the corpus as validation text and report validation loss alongside training loss. At what step does overfitting begin?
2. Train on a much larger text file (for example a Project Gutenberg novel) with STEPS = 5000. How does the output change?
3. Set LAYERS = 1 and HEADS = 1. Compare the final loss and sample quality.
4. Remove the positional embedding p["pos"] from forward(). What happens to the generated text, and why?

## 6. Decoding strategies

**File:** `l5_sampling.py`

In [7]:
"""
Layer 5 example: decoding strategies, how a model turns probabilities into text.
 
File: l5_sampling.py
 
Purpose:
    At each step an LLM outputs a score (logit) for every token in its
    vocabulary. The DECODING strategy decides which token to emit, and so
    controls how safe, varied or creative the generated text is. The
    "temperature" and "top_p" settings in LLM APIs are exactly these knobs.
 
How it works:
  greedy       always take the most likely token (repetitive, deterministic)
  temperature  divide logits by T before softmax (T<1 safer, T>1 more random)
  top-k        sample only from the k most likely tokens
  top-p        sample from the smallest set whose probabilities sum to p
               (nucleus sampling)
    The program applies each strategy to 8 made-up next-token scores for
    "The cat sat on the ..." and counts 1,000 samples from each.
 
What to look for in the output:
    - Entropy rises with temperature: 0.19 bits at T=0.2, 2.73 bits at T=3.0.
    - At T=1.0 even 'equation' is sampled once in 1,000; top-k and top-p
      remove such nonsense tokens completely.
    - Top-p adapts: it keeps 1 of 8 tokens for a confident distribution
      but 7 of 8 for an uncertain one.
 
Requires:
    NumPy.
"""
 
import numpy as np
 
# fixed seed so the sample counts are the same on every run
rng = np.random.default_rng(0)
tokens = ["mat", "sofa", "floor", "roof", "moon", "keyboard", "banana", "equation"]
# made-up scores (logits) a model might give after "The cat sat on the"
logits = np.array([4.0, 3.2, 3.0, 1.5, 0.5, 0.2, -1.0, -2.0])
 
 
# ---- Softmax and entropy ---------------------------------------------------------
# Softmax turns any scores into probabilities: exponentiate, then divide by the
# total. Subtracting the maximum first avoids overflow and changes nothing.
def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()
 
 
# Entropy = -sum p * log2(p): about 0 bits when one token is certain, and
# log2(8) = 3 bits when all 8 tokens are equally likely.
def entropy(p):
    """Uncertainty of a distribution in bits: 0 = certain."""
    p = p[p > 0]; return float(-(p * np.log2(p)).sum())
 
 
# ---- Temperature ------------------------------------------------------------------
# Dividing logits by T before softmax: small T exaggerates gaps between scores
# (sharper, nearly greedy), large T shrinks them (flatter, more random).
print("Effect of temperature on the probability distribution:")
for T in (0.2, 0.7, 1.0, 1.5, 3.0):
    p = softmax(logits / T)
    # show only the 5 most likely tokens to keep each line short
    shown = "  ".join(f"{t}:{v:.2f}" for t, v in zip(tokens[:5], p[:5]))
    print(f"  T={T:<4} entropy {entropy(p):.2f} bits   {shown}")
 
 
# ---- Top-k filter ---------------------------------------------------------------
# Zero out everything except the k most probable tokens, then renormalise so
# the survivors sum to 1. k is fixed however confident the model is.
def top_k_filter(p, k):
    keep = np.argsort(-p)[:k]
    q = np.zeros_like(p); q[keep] = p[keep]
    return q / q.sum()
 
 
# ---- Top-p (nucleus) filter -------------------------------------------------------
# Sort tokens from most to least likely and keep adding them until their
# probabilities add up to top_p. The number kept varies with the model's
# confidence, which is why top-p is the usual default in LLM APIs.
def top_p_filter(p, top_p):
    order = np.argsort(-p)                          # most likely first
    cumulative = np.cumsum(p[order])
    cutoff = np.searchsorted(cumulative, top_p) + 1      # smallest set reaching top_p
    q = np.zeros_like(p); q[order[:cutoff]] = p[order[:cutoff]]
    return q / q.sum()
 
 
# ---- Sampling ------------------------------------------------------------------------
# Draw n tokens at random according to p and count how often each appears.
def draw(p, n=1000):
    picks = rng.choice(len(p), n, p=p)
    counts = np.bincount(picks, minlength=len(p))
    return {tokens[i]: int(c) for i, c in enumerate(counts) if c}
 
 
# Compare the strategies. Greedy needs no sampling: it always picks the argmax.
p = softmax(logits)
print("\nCounts of each token over 1,000 samples:")
print("  greedy       :", {tokens[int(np.argmax(p))]: 1000})
print("  T=1.0        :", draw(p))
print("  T=0.5        :", draw(softmax(logits / 0.5)))
print("  top-k (k=3)  :", draw(top_k_filter(p, 3)))
print("  top-p (0.9)  :", draw(top_p_filter(p, 0.9)))
print("\nTop-p adapts: when the model is confident, few tokens pass; "
      "when unsure, more do.")
# Two contrasting distributions: one sharply peaked, one almost flat.
confident = softmax(np.array([8.0, 2, 1, 0.5, 0, 0, -1, -1]))
uncertain = softmax(np.array([1.0, 0.9, 0.9, 0.8, 0.8, 0.7, 0.6, 0.5]))
for name, dist in [("confident", confident), ("uncertain", uncertain)]:
    kept = int((top_p_filter(dist, 0.9) > 0).sum())
    print(f"  {name:9s} distribution: top-p 0.9 keeps {kept} of {len(tokens)} tokens")

Effect of temperature on the probability distribution:
  T=0.2  entropy 0.19 bits   mat:0.98  sofa:0.02  floor:0.01  roof:0.00  moon:0.00
  T=0.7  entropy 1.46 bits   mat:0.63  sofa:0.20  floor:0.15  roof:0.02  moon:0.00
  T=1.0  entropy 1.83 bits   mat:0.51  sofa:0.23  floor:0.19  roof:0.04  moon:0.02
  T=1.5  entropy 2.24 bits   mat:0.40  sofa:0.23  floor:0.20  roof:0.07  moon:0.04
  T=3.0  entropy 2.73 bits   mat:0.26  sofa:0.20  floor:0.19  roof:0.11  moon:0.08

Counts of each token over 1,000 samples:
  greedy       : {'mat': 1000}
  T=1.0        : {'mat': 480, 'sofa': 244, 'floor': 201, 'roof': 42, 'moon': 18, 'keyboard': 12, 'banana': 2, 'equation': 1}
  T=0.5        : {'mat': 748, 'sofa': 144, 'floor': 103, 'roof': 5}
  top-k (k=3)  : {'mat': 571, 'sofa': 228, 'floor': 201}
  top-p (0.9)  : {'mat': 557, 'sofa': 256, 'floor': 187}

Top-p adapts: when the model is confident, few tokens pass; when unsure, more do.
  confident distribution: top-p 0.9 keeps 1 of 8 tokens
  uncertain

**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Effect of temperature on the probability distribution:
  T=0.2  entropy 0.19 bits   mat:0.98  sofa:0.02  floor:0.01  roof:0.00  moon:0.00
  T=0.7  entropy 1.46 bits   mat:0.63  sofa:0.20  floor:0.15  roof:0.02  moon:0.00
  T=1.0  entropy 1.83 bits   mat:0.51  sofa:0.23  floor:0.19  roof:0.04  moon:0.02
  T=1.5  entropy 2.24 bits   mat:0.40  sofa:0.23  floor:0.20  roof:0.07  moon:0.04
  T=3.0  entropy 2.73 bits   mat:0.26  sofa:0.20  floor:0.19  roof:0.11  moon:0.08
 
Counts of each token over 1,000 samples:
  greedy       : {'mat': 1000}
  T=1.0        : {'mat': 480, 'sofa': 244, 'floor': 201, 'roof': 42, 'moon': 18, 'keyboard': 12, 'banana': 2, 'equation': 1}
  T=0.5        : {'mat': 748, 'sofa': 144, 'floor': 103, 'roof': 5}
  top-k (k=3)  : {'mat': 571, 'sofa': 228, 'floor': 201}
  top-p (0.9)  : {'mat': 557, 'sofa': 256, 'floor': 187}
 
Top-p adapts: when the model is confident, few tokens pass; when unsure, more do.
  confident distribution: top-p 0.9 keeps 1 of 8 tokens
  uncertain distribution: top-p 0.9 keeps 7 of 8 tokens
```

**What the output shows**

At temperature 1.0, "banana" and "equation" are still occasionally chosen after "The cat sat on the". Top-k and top-p remove this unreliable tail. The final lines show why top-p is often preferred: it keeps a single token when the model is confident and seven when it is unsure, whereas top-k always keeps the same number.

**Investigate**

1. Which settings would you choose for generating code, and which for brainstorming story ideas? Justify your answer.
2. Combine temperature 0.8 with top-p 0.9 and measure how often each token is chosen.
3. Implement a repetition penalty that divides the logit of any token already generated by 1.3, and apply it in the n-gram generator.

## 7. The retrieval half of RAG

**File:** `l5_rag.py`

In [8]:
"""
Layer 5 example: retrieval-augmented generation (RAG), the retrieval half.
 
File: l5_rag.py
 
Purpose:
    An LLM only knows what was in its training data, and may invent answers
    (hallucinate) about anything else. RAG answers questions using YOUR
    documents by finding the relevant passages and pasting them into the
    prompt, so the model can answer from them and cite them.
 
How it works:
  1. CHUNK documents into passages
  2. EMBED each chunk as a vector and store it (a 'vector database')
  3. EMBED the question and find the most SIMILAR chunks
  4. Put those chunks into the PROMPT and ask the LLM to answer from them
Here we use TF-IDF vectors (no downloads needed). Production systems use
neural embedding models, but steps 1 to 4 are identical.
 
What to look for in the output:
    - The staff handbook is split into 9 chunks.
    - The remote-working question finds the right chunk with similarity 0.44.
    - 'company cars' is not in the handbook, yet the VPN chunk still scores
      0.19 because it shares the word 'company': a false match. The 0.2
      threshold flags it as weak, so the assistant should say it does not know.
    - The final prompt tells the LLM to use ONLY the numbered sources.
 
Requires:
    NumPy and scikit-learn.
"""
 
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
 
# The "documents": a short staff handbook with four topics.
# (A backslash at the end of a line just continues that line of text.)
handbook = """
Annual leave. Full-time staff receive 25 days of annual leave per year plus bank \
holidays.
Leave requests must be submitted through the HR portal at least two weeks in advance.
Unused leave of up to five days may be carried over to the following year.
 
Remote working. Employees may work from home up to three days per week with manager \
approval.
A secure VPN connection must be used when accessing company systems remotely.
Home working equipment can be requested through the IT service desk.
 
Expenses. Travel expenses must be claimed within 30 days using the finance system.
Receipts are required for all claims over ten pounds. Mileage is reimbursed at 45p \
per mile.
First class rail travel is not permitted unless approved by a director.
 
IT security. Passwords must be at least 14 characters and changed if a breach is \
suspected.
Multi-factor authentication is mandatory for email and the HR portal.
Suspicious emails should be reported using the Report Phishing button in Outlook.
"""
 
# ---- 1. CHUNK: split into sentences, then group sentences into small passages -------
# Split after every full stop followed by whitespace, then join each pair of
# sentences into one chunk. Chunks must be small enough to be specific, but
# big enough to carry a complete fact.
sentences = [s.strip() for s in re.split(r"(?<=[.])\s+", handbook) if s.strip()]
chunks = [" ".join(sentences[i:i+2]) for i in range(0, len(sentences), 2)]
print(f"{len(chunks)} chunks created\n")
 
# ---- 2. EMBED and index all chunks ---------------------------------------------------
# TF-IDF: each term is weighted by how often it appears in the chunk, times how
# RARE it is across all chunks, so distinctive words count most. Common English
# words ("the", "is") are dropped; ngram_range=(1, 2) adds two-word phrases.
# A neural embedding model would instead capture meaning, e.g. "car" ~ "vehicle".
vectoriser = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
index = vectoriser.fit_transform(chunks)          # rows = chunks, columns = terms
 
 
# ---- 3. RETRIEVE: similarity search -----------------------------------------------
# Score every chunk against the question and return the k best as
# (score, text) pairs. A real vector database does this search for millions
# of chunks using approximate nearest-neighbour indexes.
def retrieve(question, k=2):
    q = vectoriser.transform([question])          # 3. embed the question the same way
    # cosine similarity (TF-IDF rows are unit length), one score per chunk
    scores = (index @ q.T).toarray().ravel()
    best = np.argsort(-scores)[:k]                # indices of the k highest scores
    return [(float(scores[i]), chunks[i]) for i in best]
 
 
# ---- 4. PROMPT ASSEMBLY ------------------------------------------------------------
# The retrieved chunks are numbered and pasted above the question. The
# instructions tell the model to stay within the sources, cite them, and admit
# ignorance: this is what reduces hallucination.
def build_prompt(question, retrieved):
    """4. The prompt that would be sent to an LLM."""
    context = "\n".join(f"[{n+1}] {text}" for n, (_, text) in enumerate(retrieved))
    return (f"Answer the question using ONLY the sources below. "
            f"Cite sources by number. "
            f"If the answer is not in the sources, say you do not know.\n\n"
            f"Sources:\n{context}\n\nQuestion: {question}\nAnswer:")
 
 
# Three test questions: one easy, one needing a small inference (8 < 10 pounds),
# and one the handbook cannot answer.
# Why a false match happens: similarity only measures shared words (or, with
# neural embeddings, shared topic), not whether the chunk answers the
# question. Some chunk always comes top, so a minimum score is needed.
for question in ["How many days can I work from home?",
                 "Do I need a receipt for a taxi costing eight pounds?",
                 "What is the policy on company cars?"]:
    results = retrieve(question)
    print(f"QUESTION: {question}")
    for score, text in results:
        print(f"  similarity {score:.2f}: {text[:85]}...")  # first 85 chars only
    # if even the best chunk scores low, it is probably a false match
    if results[0][0] < 0.2:
        print("  -> weak match (below 0.2): probably not relevant; "
              "the assistant should say it does not know")
    print()
 
print("PROMPT SENT TO THE LLM FOR THE FIRST QUESTION:\n")
# Show the full prompt an LLM would receive for the first question.
print(build_prompt("How many days can I work from home?",
                   retrieve("How many days can I work from home?")))

9 chunks created

QUESTION: How many days can I work from home?
  similarity 0.44: Remote working. Employees may work from home up to three days per week with manager a...
  similarity 0.08: A secure VPN connection must be used when accessing company systems remotely. Home wo...

QUESTION: Do I need a receipt for a taxi costing eight pounds?
  similarity 0.26: Receipts are required for all claims over ten pounds. Mileage is reimbursed at 45p pe...
  similarity 0.00: Annual leave. Full-time staff receive 25 days of annual leave per year plus bank holi...

QUESTION: What is the policy on company cars?
  similarity 0.19: A secure VPN connection must be used when accessing company systems remotely. Home wo...
  similarity 0.00: Annual leave. Full-time staff receive 25 days of annual leave per year plus bank holi...
  -> weak match (below 0.2): probably not relevant; the assistant should say it does not know

PROMPT SENT TO THE LLM FOR THE FIRST QUESTION:

Answer the question using ONLY the

**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
9 chunks created
 
QUESTION: How many days can I work from home?
  similarity 0.44: Remote working. Employees may work from home up to three days per week with manager a...
  similarity 0.08: A secure VPN connection must be used when accessing company systems remotely. Home wo...
 
QUESTION: Do I need a receipt for a taxi costing eight pounds?
  similarity 0.26: Receipts are required for all claims over ten pounds. Mileage is reimbursed at 45p pe...
  similarity 0.00: Annual leave. Full-time staff receive 25 days of annual leave per year plus bank holi...
 
QUESTION: What is the policy on company cars?
  similarity 0.19: A secure VPN connection must be used when accessing company systems remotely. Home wo...
  similarity 0.00: Annual leave. Full-time staff receive 25 days of annual leave per year plus bank holi...
  -> weak match (below 0.2): probably not relevant; the assistant should say it does not know
 
PROMPT SENT TO THE LLM FOR THE FIRST QUESTION:
 
Answer the question using ONLY the sources below. Cite sources by number. If the answer is not in the sources, say you do not know.
 
Sources:
[1] Remote working. Employees may work from home up to three days per week with manager approval.
[2] A secure VPN connection must be used when accessing company systems remotely. Home working equipment can be requested through the IT service desk.
 
Question: How many days can I work from home?
Answer:
```

**What the output shows**

The first two questions retrieve exactly the right passages, even though the second uses different words ("receipt", "taxi") from the handbook. The third question exposes a weakness of keyword retrieval: "company cars" matches the unrelated VPN passage because both contain "company". A similarity threshold, a better embedding model and an instruction to say "I do not know" when sources are insufficient are all needed in practice.

**Investigate**

1. Add a paragraph about company car policy to the handbook and rerun the third question.
2. Change the chunk size from 2 sentences to 1 and to 4. How does retrieval quality change for each question?
3. If you have an API key, send the generated prompt to an LLM using the code in Section 6.11 and check whether the answer cites the correct source.

## 8. A diffusion model generating 2D data

**File:** `l5_diffusion.py`

In [9]:
"""
Layer 5 example: a denoising diffusion model (Ho et al., 2020)
generating 2D points, the same principle used by image generators.
 
File: l5_diffusion.py
 
Purpose:
    Image generators such as Stable Diffusion and DALL-E learn to turn random
    noise into realistic images by removing noise a little at a time. Here
    the "images" are just 2D points shaped like two interlocking moons, so
    the whole idea fits in a few seconds of CPU time.
 
How it works:
FORWARD process (fixed): gradually add Gaussian noise over 50 steps until the
data becomes pure noise:   x_t = sqrt(abar_t) * x_0 + sqrt(1 - abar_t) * noise
 
TRAINING: teach a neural network to look at a noisy point x_t and the step t,
and predict the noise that was added.
 
GENERATION (reverse process): start from pure random noise and repeatedly
subtract the predicted noise, step by step, until a realistic sample emerges.
 
What to look for in the output:
    - The signal fades from 1.0 at step 1 to 0.068 at step 50: almost pure noise.
    - Generated points lie 0.048 from the nearest real point on average,
      close to real held-out data (0.030) and far better than noise (0.287).
    - 48% of generated points fall on the left half (real data: 50%), so
      the model reproduces BOTH moons rather than collapsing onto one.
 
Requires:
    NumPy and scikit-learn.
"""
 
import numpy as np
import warnings
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore")                  # hide convergence warnings
 
rng = np.random.default_rng(0)                      # fixed seed: repeatable run
# The "real data" we want to learn to generate: 3,000 points on two moons,
# shape (3000, 2). Real diffusion models use images with millions of values.
X0, _ = make_moons(3000, noise=0.05, random_state=0)
X0 = (X0 - X0.mean(0)) / X0.std(0)                 # standardise the 'real data'
 
# ---- Noise schedule ------------------------------------------------------------------
# beta_t = fraction of variance replaced by noise at step t. It starts tiny
# and grows, so early steps barely change the data and late steps destroy it.
# abar_t (alpha-bar) multiplies all alphas up to t, so x_t can be made from x_0
# in ONE jump instead of t separate noising steps.
STEPS = 50
betas = np.linspace(1e-4, 0.2, STEPS)               # noise added at each step
alphas = 1 - betas
abar = np.cumprod(alphas)                  # total signal remaining after t steps
# sqrt(abar_t) is the weight on the original data in x_t (1 = untouched)
print("signal remaining at step 1, 10, 25, 50:",
      np.round(np.sqrt(abar[[0, 9, 24, 49]]), 3))
 
# ---- Build a training set of (noisy point, step) -> noise ------------------------
# Each real point is reused 20 times, each time noised to a random step t.
# The network sees (x_t, t) and must output the exact noise vector used.
reps = 20
x0 = np.repeat(X0, reps, axis=0)                    # shape (60000, 2)
t = rng.integers(0, STEPS, len(x0))                 # a random step per example
noise = rng.normal(size=x0.shape)
# forward process in one jump: blend the clean point with fresh noise
xt = np.sqrt(abar[t])[:, None] * x0 + np.sqrt(1 - abar[t])[:, None] * noise
# time embedding: describe the step t with 3 smooth numbers, shape (60000, 3),
# so the network knows how noisy its input is
t_feat = np.stack([np.sin(t / STEPS * np.pi), np.cos(t / STEPS * np.pi),
                   t / STEPS], 1)
 
# The DENOISER: a 3-layer neural network. Input = 2 coordinates + 3 time
# features; output = the 2 predicted noise values. It is trained with mean
# squared error. Image models use a U-Net or transformer instead.
model = MLPRegressor(hidden_layer_sizes=(128, 128, 128), max_iter=60, batch_size=512,
                     learning_rate_init=2e-3, random_state=0)
model.fit(np.hstack([xt, t_feat]), noise)
print("noise-prediction network trained on", len(x0), "examples")
 
 
# ---- Reverse sampling loop (generation) -------------------------------------------
# Run the chain backwards from step 49 to step 0. At each step the network
# guesses the noise in x, we subtract a scaled amount of it, and (except at
# the last step) add a little fresh randomness so samples stay varied.
def sample(n):
    x = rng.normal(size=(n, 2))                     # start from PURE NOISE
    for step in reversed(range(STEPS)):
        # the same time embedding as in training, copied for all n points
        tf = np.tile([np.sin(step / STEPS * np.pi), np.cos(step / STEPS * np.pi),
                      step / STEPS], (n, 1))
        eps = model.predict(np.hstack([x, tf]))    # network's guess of the noise
        # DDPM update: remove the predicted noise, then add a little fresh noise
        # (except at the end)
        x = (x - betas[step] / np.sqrt(1 - abar[step]) * eps) / np.sqrt(alphas[step])
        if step > 0:
            x += np.sqrt(betas[step]) * rng.normal(size=x.shape)
    return x
 
 
# ---- Evaluation -------------------------------------------------------------------
# A good generator produces points that lie ON the moons. We measure the
# average distance to the closest of 1,000 real points (every third one).
def moon_distance(points):
    """Average distance from each point to its nearest REAL data point."""
    # pairwise distances by broadcasting: (n, 1, 2) - (1, 1000, 2) -> (n, 1000)
    d = np.sqrt(((points[:, None, :] - X0[None, ::3, :]) ** 2).sum(-1))
    return d.min(1).mean()
 
 
# Compare generated samples with the noise they started from and with
# fresh real data (the best score we could hope for).
generated = sample(1000)
pure_noise = rng.normal(size=(1000, 2))
print(f"\naverage distance to nearest real point:")
print(f"  pure Gaussian noise : {moon_distance(pure_noise):.3f}")
print(f"  diffusion samples   : {moon_distance(generated):.3f}")
# rebuild the original scaling so new real data is standardised the same way
real_mean, real_std = (make_moons(3000, noise=0.05, random_state=0)[0].mean(0),
                       make_moons(3000, noise=0.05, random_state=0)[0].std(0))
# new real data, same scaling
held_out = (make_moons(1000, noise=0.05, random_state=1)[0] - real_mean) / real_std
print(f"  real held-out data  : {moon_distance(held_out):.3f}")
# "Mode collapse" check: are both moons generated in the right proportion?
left = (generated[:, 0] < 0).mean()
print(f"\nshare of generated points on the left half: {left:.2f} "
      f"(real data: {(X0[:, 0] < 0).mean():.2f})")

signal remaining at step 1, 10, 25, 50: [1.    0.911 0.53  0.068]


noise-prediction network trained on 60000 examples

average distance to nearest real point:
  pure Gaussian noise : 0.287


  diffusion samples   : 0.048
  real held-out data  : 0.030

share of generated points on the left half: 0.48 (real data: 0.50)


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
signal remaining at step 1, 10, 25, 50: [1.    0.911 0.53  0.068]
noise-prediction network trained on 60000 examples
 
average distance to nearest real point:
  pure Gaussian noise : 0.287
  diffusion samples   : 0.048
  real held-out data  : 0.030
 
share of generated points on the left half: 0.48 (real data: 0.50)
```

**What the output shows**

Starting from shapeless random noise, 50 denoising steps produce points that trace the two crescent shapes of the training data, with the balance between the two halves preserved. Image diffusion models apply exactly this procedure to millions of pixel values, using a U-Net or transformer as the noise predictor.

**Investigate**

1. Reduce STEPS to 10. How does sample quality change?
2. Replace make_moons with make_circles and retrain. Does the model learn the new shape?
3. Save the intermediate x at steps 40, 25 and 10 during sampling and plot them to watch the structure emerge.

## 9. L5_llm_apis.py

**File:** `l5_llm_apis.py`

The listing below shows three ways to work with real models: running GPT-2 locally with the Hugging Face Transformers library (Hugging Face, n.d.), calling a hosted model through the Anthropic API (Anthropic, n.d.a), and running an open model on your own machine with Ollama. These require downloads, an internet connection or an API key, so they were not run in the environment used to build this guide. Model names and SDK details change frequently; check current documentation first.

> **Not run in this notebook.** This program needs the transformers and torch packages, an Anthropic API key and a local Ollama server. Install what it needs and run it on your own machine; the code is included so that you can read and adapt it.

In [ ]:
"""
Layer 5 example: using real generative models through frameworks and APIs.
 
File: l5_llm_apis.py
 
Purpose:
    The other Layer 5 programs build tokenisers, language models and samplers
    from scratch. This listing shows how the same ideas appear when you use
    real models: BPE tokens, next-token probabilities, decoding settings
    (temperature, top-p, top-k), and a chat API with roles and a system prompt.
 
    Layer 5 framework examples: using real generative models.
    Not executed in the environment used to build this guide (no internet model
    downloads or API key). Model names and SDK details change frequently:
    check the current documentation before running.
 
How it works:
  Part 1  GPT-2 locally with Hugging Face Transformers   pip install transformers torch
  Part 2  A hosted LLM through the Anthropic Python SDK   pip install anthropic
  Part 3  An open model running locally with Ollama       https://ollama.com
 
What to look for in the output:
    No reference output: this file was not run. If you run it, expect to see
    GPT-2's BPE pieces (words often start with a marker for a leading space),
    the five most likely next tokens with their probabilities, three
    continuations that differ in how adventurous they are, a short tutor-style
    answer from the hosted model with its input and output token counts, and
    a haiku from the local model.
 
Requires:
    Part 1: transformers and torch (downloads GPT-2, about 500 MB).
    Part 2: anthropic, plus an API key in the ANTHROPIC_API_KEY environment
            variable (never write the key into the code).
    Part 3: requests, plus Ollama installed and running on this computer.
"""
 
# ===================== Part 1: Hugging Face Transformers =====================
# GPT-2 is small enough to run on a laptop. It is a decoder-only transformer,
# the same design as the mini GPT in l5_mini_gpt.py, just much larger.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
 
# Download (the first time) the tokeniser and the trained weights of the
# 124M-parameter GPT-2 (2019).
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
 
# Turn the prompt into token ids, shape (1, number_of_tokens); "pt" = PyTorch.
prompt = "The most important skill for a software engineer is"
ids = tokenizer(prompt, return_tensors="pt").input_ids
print("tokens:", tokenizer.convert_ids_to_tokens(ids[0]))  # see the BPE pieces
 
# Inspect the model's probability distribution for the NEXT token
# no_grad(): we are only predicting, not training, so skip gradient tracking.
# logits has shape (1, tokens, vocabulary); [0, -1] keeps the scores for the
# position after the last prompt token, one per vocabulary entry (50,257).
with torch.no_grad():
    logits = model(ids).logits[0, -1]
probs = torch.softmax(logits, dim=-1)       # scores -> probabilities summing to 1
top = torch.topk(probs, 5)                  # the 5 most likely next tokens
for p, i in zip(top.values, top.indices):
    print(f"  {tokenizer.decode(i)!r:15s} {p:.3f}")
 
# Generate with different decoding strategies
# These are the settings demonstrated in l5_sampling.py. max_new_tokens caps
# the length of the continuation; GPT-2 has no padding token, so the
# end-of-text token is used instead to silence a warning.
for settings in [dict(do_sample=False),                             # greedy
                 dict(do_sample=True, temperature=0.7, top_p=0.9),  # nucleus sampling
                 dict(do_sample=True, temperature=1.5, top_k=50)]:  # high temperature
    out = model.generate(ids, max_new_tokens=30,
                         pad_token_id=tokenizer.eos_token_id, **settings)
    # out holds the prompt followed by the new tokens; print only the new part
    print(settings, "->", tokenizer.decode(out[0][ids.shape[1]:]))
 
# ===================== Part 2: Anthropic API ================================
# A hosted model runs on the provider's servers: you send text, it returns
# text, and you pay per token. The API key identifies (and bills) you, so it is
# read from an environment variable and must never be written into code.
import anthropic
 
# reads the ANTHROPIC_API_KEY environment variable
client = anthropic.Anthropic()
# One request = one call to messages.create:
#   max_tokens   hard limit on the length of the reply (long replies are cut off)
#   temperature  lower = more focused and repeatable answers
#   system       the SYSTEM PROMPT: instructions about role, tone and rules that
#                apply to the whole conversation, kept separate from user input
#   messages     the conversation so far; each message has a ROLE ("user" for
#                the person, "assistant" for the model's earlier replies)
response = client.messages.create(
    model="claude-sonnet-5",            # check docs.claude.com for current model names
    max_tokens=300,
    temperature=0.3,
    system=("You are a patient tutor for Level 3 computing students. "
            "Answer in under 100 words."),
    messages=[{"role": "user",
               "content": "Explain what a token is in a large language model."}],
)
# The reply is a list of content blocks; the first one holds the text.
print(response.content[0].text)
# Usage counts show what the request cost: both prompt and reply are billed.
print("input tokens:", response.usage.input_tokens,
      "output tokens:", response.usage.output_tokens)
 
# ===================== Part 3: Ollama (local open model) ====================
# After installing Ollama and running:  ollama pull llama3.2
# Ollama runs an open-weights model on your own machine and offers a small web
# API on port 11434. No API key is needed and no data leaves your computer.
import requests
 
# "stream": False waits for the full answer instead of receiving it in pieces.
r = requests.post("http://localhost:11434/api/generate",
                  json={"model": "llama3.2", "prompt": "Write a haiku about debugging.",
                        "stream": False, "options": {"temperature": 0.8}})
print(r.json()["response"])

**Investigate**

1. Compare GPT-2's top-5 next-token probabilities for your own prompt with the distribution in the sampling example.
2. Send the same question to a hosted model with temperature 0 three times, then with temperature 1 three times, and compare the consistency of the answers.
3. Ask a model for three academic references on a narrow topic, then verify each one exists. Record how many were accurate.